# Make data for getting CPI weight

In [1]:
import polars as pl

In [13]:
map_month = pl.DataFrame(
    {
        "ThaiMonthAbbrName": [
            'ม.ค.', 'ก.พ.', 'มี.ค.', 'เม.ย.', 'พ.ค.', 'มิ.ย.',
            'ก.ค.', 'ส.ค.', 'ก.ย.', 'ต.ค.', 'พ.ย.', 'ธ.ค.'
        ],
        "Month": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
    }
)

In [ ]:
consumption = pl.read_excel(
    'CPIG.xlsx', sheet_name='รหัสหมวด', has_header=False
) \
    .slice(4) \
    .rename({'column_1': 'ConsumptionKey', 'column_2': 'ThaiConsumptionName'}) \
    .drop_nulls('ConsumptionKey') \
    .with_columns([
        pl.col('ConsumptionKey').str.strip_chars(),
        pl.col('ThaiConsumptionName').str.strip_chars(),
    ])

cpi = pl.read_excel(
    'CPIG.xlsx', 
    sheet_name='Index (ของเดือน)', 
    read_options={
        "header_row": 3
    }
) \
    .rename(
        {
            'หมวด': 'ThaiConsumptionName', 
            'ปีฐาน': 'BaseYear',
            'ปี': "Year"
        }
    ) \
    .cast({"BaseYear": pl.Int64, "Year": pl.Int64}) \
    .with_columns(
        pl.col('ThaiConsumptionName').str.strip_chars(),
        (pl.col('BaseYear') - 543),
        (pl.col('Year') - 543)
    )

In [ ]:
cpi_df = cpi \
    .join(consumption, on='ThaiConsumptionName', how='left') \
    .drop("ThaiConsumptionName") \
    .unpivot(
        on=None,
        index=['ConsumptionKey', 'BaseYear', 'Year'],
        variable_name='ThaiMonthAbbrName',
        value_name='ConsumerPriceIndex'
    ) \
    .drop_nulls('ConsumerPriceIndex') \
    .join(map_month, on='ThaiMonthAbbrName', how='left') \
    .with_columns(
        pl.date(pl.col('Year'), pl.col('Month'), 1).alias("Date"),
        pl.col("ConsumerPriceIndex").str.replace_all(',', ""),
        ("var" + pl.col("ConsumptionKey").str.strip_chars_end("0").replace("", "0")).alias("ConsumptionKey")
    ) \
    .cast({"ConsumerPriceIndex": pl.Float64}) \
    .pivot(on="ConsumptionKey", index=["Date", "BaseYear"], values="ConsumerPriceIndex", aggregate_function="first") \
    .sort("Date")

In [42]:
cpi_df.write_csv("cpi_df.csv")